# 03 · Comparing Models &amp; Visualizing Phonemes

Quantitative comparison with `zipa-cli compare`, programmatic alignment, producing
`align-json` for the web viewer, and a quick matplotlib phone-timeline plot.

In [ ]:
# Make the zipa_cli package importable when running from tutorials/ without install.
import sys, os
REPO = os.path.abspath('..')
if REPO not in sys.path: sys.path.insert(0, REPO)
from zipa_cli.cli import main as zipa  # call like zipa(['models','list'])
print('zipa-cli importable from', REPO)

In [ ]:
# Which backend deps are available in this kernel?
def have(mod):
    try:
        __import__(mod); return True
    except Exception:
        return False
ONNX_OK = all(have(m) for m in ['onnxruntime','soundfile','lhotse'])
print('ONNX inference available:', ONNX_OK)
print('(install with: pip install -e "..[onnx]")')

In [ ]:
# Build a tiny *valid* ONNX CTC model + tokens so this notebook runs end-to-end
# WITHOUT downloading the multi-hundred-MB ZIPA checkpoints. Real usage just swaps
# --model <path> for a registry tag like 'zipa-cr-s-300k'.
import numpy as np, tempfile
TMP = tempfile.mkdtemp(prefix='zipa_demo_')
def build_tiny_ctc():
    import onnx
    from onnx import helper, TensorProto
    V = 10
    W = (np.random.RandomState(0).randn(80, V)*0.1).astype(np.float32)
    g = helper.make_graph([helper.make_node('MatMul',['x','W'],['logits'])],'tiny',
        [helper.make_tensor_value_info('x',TensorProto.FLOAT,['B','T',80]),
         helper.make_tensor_value_info('x_lens',TensorProto.INT64,['B'])],
        [helper.make_tensor_value_info('logits',TensorProto.FLOAT,['B','T',V])],
        [helper.make_tensor('W',TensorProto.FLOAT,[80,V],W.flatten())])
    m = helper.make_model(g, opset_imports=[helper.make_opsetid('',13)]); m.ir_version=9
    p = os.path.join(TMP,'tiny_ctc.onnx'); onnx.save(m,p)
    tk = os.path.join(TMP,'tokens.txt')
    with open(tk,'w') as f:
        for i,t in enumerate('<blk> a b c d e f g h i'.split()): f.write(f'{t} {i}\n')
    return p, tk
MODEL = TOKENS = None
if ONNX_OK and have('onnx'):
    MODEL, TOKENS = build_tiny_ctc()
    print('demo model:', MODEL)
else:
    print('Skipping demo model (need onnxruntime+lhotse+onnx). Markdown cells still apply.')

## Programmatic alignment (no heavy deps)
`zipa_cli.analysis.align` gives Match/Sub/Ins/Del between two phone sequences.

In [ ]:
from zipa_cli.analysis import align, compare
counts, ops = align(['a','b','c','d'], ['a','x','c'])
print('match',counts.match,'sub',counts.sub,'ins',counts.ins,'del',counts.delete,'PER',round(counts.per,3))
for kind, r, h in ops: print(f'{kind:5} ref={r!r:>4} hyp={h!r}')

## `zipa-cli compare` on two transcript files

In [ ]:
A=os.path.join(TMP,'A.tsv'); B=os.path.join(TMP,'B.tsv')
open(A,'w').write('u1\ta b c d\nu2\tp q\n')
open(B,'w').write('u1\ta x c\nu2\tp q r\n')
zipa(['compare','--a',A,'--b',B,'--no-pfer','--top-k','5'])

> Install `..[analysis]` (panphon) to also get articulatory-feature PFER.

## Produce `align-json` for the web viewer

In [ ]:
if MODEL:
    SAMPLE=os.path.join(REPO,'zipa','sample.wav')
    aj=os.path.join(TMP,'aligned.json')
    zipa(['decode','--input',SAMPLE,'--input-type','file','--model',MODEL,'--model-type','ctc',
          '--tokens',TOKENS,'-o',aj,'--output-format','align-json'])
    import json; rec=json.loads(open(aj).readline())
    print('phones:', rec['phones'][:6], '...'); ALIGN=rec
else:
    # synthetic alignment so the plot below still works without onnx
    ALIGN={'id':'demo','phones':[{'p':p,'start':i*0.2,'end':i*0.2+0.18} for i,p in enumerate('h e l o w'.split())]}
    print('using synthetic alignment')

## Quick phone-timeline plot

In [ ]:
try:
    import matplotlib.pyplot as plt
    ph=ALIGN['phones']
    fig,ax=plt.subplots(figsize=(10,1.6))
    for d in ph:
        ax.barh(0, d['end']-d['start'], left=d['start'], height=0.6, edgecolor='k')
        ax.text((d['start']+d['end'])/2, 0, d['p'], ha='center', va='center')
    ax.set_yticks([]); ax.set_xlabel('time (s)'); ax.set_title('Decoded phones')
    plt.tight_layout(); plt.show()
except ImportError:
    print('pip install matplotlib to see the timeline plot')

## Open the interactive web viewer
For waveform/spectrogram + multi-model tiers:
```bash
# one align-json per model, then serve web/
zipa-cli decode --input utt.wav --model zipa-cr-l-500k --output-format align-json -o large.json
zipa-cli decode --input utt.wav --model zipa-cr-s-300k --output-format align-json -o small.json
cd web && python -m http.server 8000   # http://localhost:8000
```
Load the audio and both JSON files; tiers stack per model. See `web/README.md`.